# PhysioNet-109 source-only LOSO — local GPU, 54 folds per run

Runs the same experiment as `kaggle_train_source_only_physionet109_loso_single_gpu_bs144.ipynb`
(pure source-only, batch 144, 75 epochs, no RA, no IM-TTA) on your own GPU machine, without the
Kaggle 12-hour limit.

* `PART = 1` evaluates targets S001–S054 (54 folds) in one run; `PART = 2` evaluates S055–S109 (55 folds).
  Every fold trains on the other 108 subjects.
* `GPU_ID` selects the CUDA device; `WORK_DIR` (or the `PHYSIONET_WORK_DIR` environment variable)
  holds the repository clone, the PhysioNet cache (~6 runs × 109 subjects from the AWS mirror) and the results.
* Requirements: `git`, Python ≥ 3.8 to run this notebook, an NVIDIA driver recent enough for CUDA 12.6
  wheels. The notebook creates an isolated Python 3.10 environment with `uv`.
* `results.txt` is written when all folds of the part have finished.


In [ ]:
# Full-Mamba HADA - PhysioNet 109-subject source-only LOSO on a local GPU machine.
# One run evaluates 54 target folds back to back (PART = 1: S001-S054, PART = 2: S055-S109).
# Same settings as kaggle_train_source_only_physionet109_loso_single_gpu_bs144.ipynb
# (batch 144, 75 epochs, pure source-only, no RA, no IM-TTA); only the Kaggle-specific
# paths, GPU checks and output handling are replaced. Works on Linux and Windows.
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

# ---------------- Run configuration ----------------
PART = 1               # 1 -> targets S001-S054 (54 folds); 2 -> targets S055-S109 (55 folds).
GPU_ID = 0             # CUDA device index on this machine.
MAX_EPOCHS = 75
BATCH_SIZE = 144
USE_RA = False
IM_TTA_STEPS = 0
# Working directory for the repository clone, the PhysioNet cache and the results.
WORK_DIR = Path(os.environ.get('PHYSIONET_WORK_DIR', Path.cwd() / 'physionet109_local')).resolve()
# PyTorch CUDA wheel index; cu126 runs on any recent NVIDIA driver (>= 560 on Linux).
TORCH_INDEX = 'https://download.pytorch.org/whl/cu126'

ALL_SUBJECTS = list(range(1, 110))
TARGET_SUBJECTS = {1: ALL_SUBJECTS[:54], 2: ALL_SUBJECTS[54:]}[PART]
JOB_TAG = f's{TARGET_SUBJECTS[0]:03d}_s{TARGET_SUBJECTS[-1]:03d}'

DATASET = 'physionet'
BRANCH = 'feature/hada-full-mamba-no-da'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
RUN_TAG = f'{DATASET}109_source_only_local_bs{BATCH_SIZE}_ep{MAX_EPOCHS}_{JOB_TAG}'
REPO_PATH = WORK_DIR / 'tcformer-physionet109'
MNE_DATA = WORK_DIR / 'mne_data'
TRAIN_LOG = WORK_DIR / f'{RUN_TAG}.log'
RESULT_ARCHIVE = WORK_DIR / f'{RUN_TAG}_results'
WINDOWS = platform.system() == 'Windows'
VENV_BIN = REPO_PATH / '.venv' / ('Scripts' if WINDOWS else 'bin')
PYTHON = VENV_BIN / ('python.exe' if WINDOWS else 'python')
AWS = VENV_BIN / ('aws.exe' if WINDOWS else 'aws')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f'PART {PART}: {len(TARGET_SUBJECTS)} folds, targets S{TARGET_SUBJECTS[0]:03d}-'
      f'S{TARGET_SUBJECTS[-1]:03d}; working directory {WORK_DIR}', flush=True)


def run(command, cwd=None, env=None, log_path=None):
    """Run a command; with log_path, stream its output line by line to the notebook and the log."""
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    with open(log_path, 'w', encoding='utf-8') as log_file:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Training exited with code {return_code}. Full log: {log_path}')


def run_with_retries(command, attempts=5, cwd=None):
    for attempt in range(1, attempts + 1):
        try:
            run(command, cwd=cwd)
            return
        except subprocess.CalledProcessError:
            if attempt == attempts:
                raise
            delay = min(15 * attempt, 60)
            print(f'Command failed (attempt {attempt}/{attempts}); retrying in {delay}s ...', flush=True)
            time.sleep(delay)


# ---------------- Clone the branch ----------------
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

# The base branch uses S001-S020 only. Expand the source pool to all 109 subjects,
# exactly as the Kaggle bs144 notebook does; TARGET_SUBJECTS are the folds scored here.
physionet_path = REPO_PATH / 'datamodules/physionet.py'
physionet_text = physionet_path.read_text(encoding='utf-8')
replacements = {
    'First-20-subject PhysioNet': 'Full-109-subject PhysioNet',
    'Each of the 19': 'Each of the 108',
    'all_subject_ids = list(range(1, 21))': 'all_subject_ids = list(range(1, 110))',
    'target subject must be in 1..20': 'target subject must be in 1..109',
    'Materializing all 20 subjects': 'Materializing all 109 subjects',
    'did not materialize subjects 1..20': 'did not materialize subjects 1..109',
    'if len(source_ids) != 19': 'if len(source_ids) != 108',
}
for old, new in replacements.items():
    if old not in physionet_text:
        raise RuntimeError(f'Expected PhysioNet source marker is missing: {old!r}')
    physionet_text = physionet_text.replace(old, new)
physionet_path.write_text(physionet_text, encoding='utf-8')
assert 'all_subject_ids = list(range(1, 110))' in physionet_text
assert 'if len(source_ids) != 108' in physionet_text
pipeline_text = (REPO_PATH / 'train_pipeline.py').read_text(encoding='utf-8')
model_text = (REPO_PATH / 'models/hada_tcformer.py').read_text(encoding='utf-8')
assert 'ModelCheckpoint' in pipeline_text and 'monitor="val_acc"' in pipeline_text
assert 'ordinary labeled' in model_text and 'source batch' in model_text
print('Verified: 109-subject source pool, source-only, and source-validation checkpointing.')

# ---------------- Environment (isolated venv, pinned versions) ----------------
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv') or [sys.executable, '-m', 'uv']
UV = UV if isinstance(UV, list) else [UV]
if not PYTHON.exists():
    run(UV + ['venv', '--python', '3.10', '.venv'], cwd=REPO_PATH)
run(UV + ['pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
          '--index-url', TORCH_INDEX])
run(UV + ['pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run(UV + ['pip', 'install', '--python', PYTHON, 'awscli'])

# ---------------- Resumable PhysioNet prefetch (AWS mirror) ----------------
eegbci_cache = MNE_DATA / 'MNE-eegbci-data/files/eegmmidb/1.0.0'
eegbci_cache.mkdir(parents=True, exist_ok=True)
aws_sync = [AWS, 's3', 'sync', 's3://physionet-open/eegmmidb/1.0.0/', eegbci_cache,
            '--no-sign-request', '--only-show-errors', '--exclude', '*']
for run_id in (4, 6, 8, 10, 12, 14):
    aws_sync.extend(['--include', f'S*/S*R{run_id:02d}.edf'])
run_with_retries(aws_sync, attempts=5)
cached_edfs = list(eegbci_cache.glob('S*/S*R*.edf'))
expected_edfs = len(ALL_SUBJECTS) * 6
if len(cached_edfs) != expected_edfs:
    raise RuntimeError(f'PhysioNet prefetch incomplete: {len(cached_edfs)}/{expected_edfs} EDF files')
print(f'PhysioNet prefetch complete: {len(cached_edfs)}/{expected_edfs} EDF files')

# ---------------- Source-only, batch 144, 75 epochs ----------------
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    f"c['max_epochs_loso']={MAX_EPOCHS}; "
    "c['log_every_n_epochs']=10; c['enable_progress_bar']=False; "
    "c['preprocessing']['physionet']['trial_duration']=4.0; "
    f"c['preprocessing']['physionet']['batch_size']={BATCH_SIZE}; "
    "c['preprocessing']['physionet']['accumulate_grad_batches']=1; "
    "c['preprocessing']['physionet']['num_workers']=2; "
    f"c['preprocessing']['physionet']['riemannian_alignment']={USE_RA!r}; "
    f"c['preprocessing']['physionet'].setdefault('model_overrides', {{}})['im_tta_steps']={IM_TTA_STEPS}; "
    # Pure source-only: no target batches, no adversarial or MMD terms.
    "c['domain_adaptation']=False; "
    "c['model_kwargs'].pop('feature_alignment', None); "
    "c['model_kwargs']['adversarial_weight']=0.0; "
    "c['model_kwargs']['mmd_weight']=0.0; "
    "c['model_kwargs']['temporal_mmd_weight']=0.0; "
    "c['model_kwargs']['log_every_n_batches']=0; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, '-c', override], cwd=REPO_PATH)
if shutil.which('nvidia-smi'):
    run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); "
     "print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'No CUDA GPU visible to PyTorch'; "
     f"print('GPU:', torch.cuda.get_device_name({GPU_ID}))"])

# ---------------- Train / test the 54 (or 55) folds ----------------
results_dir = REPO_PATH / 'results'
if results_dir.exists():
    shutil.rmtree(results_dir)
environment = os.environ.copy()
environment.update({
    'PYTHONUNBUFFERED': '1',
    'PYTHONIOENCODING': 'utf-8',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'MPLBACKEND': 'Agg',
    'MNE_DATA': str(MNE_DATA),
    'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA),
})
print(f'===== SOURCE-ONLY FULL-MAMBA HADA | PHYSIONET 109-SUBJECT LOSO | LOCAL GPU {GPU_ID} '
      f'| PART {PART} | {len(TARGET_SUBJECTS)} FOLDS S{TARGET_SUBJECTS[0]:03d}-S{TARGET_SUBJECTS[-1]:03d} '
      f'| BATCH={BATCH_SIZE} | {MAX_EPOCHS} EPOCHS | 4 CLASSES | 64 CH =====', flush=True)
print('Protocol: each fold trains only on 108 labeled source subjects; target EEG and labels are '
      'excluded from training and adaptation, then used only for final scoring.', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
started = time.time()
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', DATASET,
     '--loso', '--gpu_id', str(GPU_ID)], cwd=REPO_PATH, env=environment, log_path=TRAIN_LOG)

archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print(f'Completed {len(TARGET_SUBJECTS)} folds in {(time.time() - started) / 3600:.1f} h')
print('Result archive:', archive)
print('Training log:', TRAIN_LOG)
